%% [markdown]<br>
# BERT for Sentiment Analysis => fine-tuning a pre-trained BERT

%%<br>
!pip install transformers -U tensorflow

%%

In [ ]:
import transformers
import json
from collections import Counter
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from torch.utils.data import TensorDataset, DataLoader

Support GPU

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

%% [markdown]<br>
# Downloading large review movie dataset (25000 reviews)

%%<br>
!wget https://thome.isir.upmc.fr/classes/RITAL/json_pol.json

%%<br>
Loading json

In [ ]:
file = './json_pol.json'
with open(file, encoding="utf-8") as f:
    data = json.load(f)

Quick Check

In [ ]:
counter = Counter((x[1] for x in data))
print("Number of reviews : ", len(data))
print("----> # of positive : ", counter[1])
print("----> # of negative : ", counter[0])
print("")
print("Example:", data[0])

%% [markdown]<br>
# Getting the Tokenizer

%%

In [ ]:
model_name = "haisongzhang/roberta-tiny-cased"
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)

%% [markdown]<br>
# Let's tokenize the whole dataset

%%

In [ ]:
maxL = 512
inputs_tokens = []
attention_masks = []
labels =[]

In [ ]:
print("Tokenizing data...")
for i in range(len(data)):
    if i > 0 and i % 5000 == 0:
        print(f"Processed {i} reviews...")
        
    string_tokenized = tokenizer.encode_plus(data[i][0], return_tensors="pt",
                                             add_special_tokens=True,
                                             max_length=maxL,
                                             truncation=True,
                                             padding='max_length',
                                             return_attention_mask=True)
    inputs_tokens.append(string_tokenized['input_ids'])
    attention_masks.append(string_tokenized['attention_mask'])
    labels.append(data[i][1])

%% [markdown]<br>
# Let's create a 'TensorDataSet'

%%<br>
Converting input tokens to torch tensors

In [ ]:
inputs_tokens = torch.cat(inputs_tokens, dim=0)
attention_masks = torch.cat(attention_masks, dim=0)

Converting labels to torch tensor - IMPORTANT : CrossEntropy attend des "long"

In [ ]:
y = torch.tensor(labels, dtype=torch.long)

In [ ]:
print("Inputs shape:", inputs_tokens.shape)
print("Masks shape:", attention_masks.shape)
print("Labels shape:", y.shape)

%%

In [ ]:
np.random.seed(0)
rs = 10

Splitting Data

In [ ]:
inputs_tokens_train, inputs_tokens_test, attention_masks_train, attention_masks_test, y_train, y_test = train_test_split(
    inputs_tokens, attention_masks, y, test_size=0.5, random_state=rs
)

%%

In [ ]:
dataset_train = TensorDataset(inputs_tokens_train, attention_masks_train, y_train)
dataset_test = TensorDataset(inputs_tokens_test, attention_masks_test, y_test)

%%[markdown]<br>
# Lets download a BERT model for word embedding

%%

In [ ]:
from transformers import AutoModelForSequenceClassification

Il faut prÃ©ciser num_labels=2 pour de la classification binaire (Positif/NÃ©gatif)

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)
model.to(device)

%% [markdown]<br>
# FINE-TUNING THE MODEL

%%

In [ ]:
import gc
gc.collect()
torch.cuda.empty_cache()

%%<br>
Fonction accuracy corrigÃ©e : Elle s'adapte dÃ©sormais Ã  la taille rÃ©elle du Dataset

In [ ]:
def accuracy(model, dataloader):
    model.eval()
    nbgood = 0
    total = 0
    for idx, batch in enumerate(dataloader):
        b_input_ids = batch[0].to(device)
        b_input_mask = batch[1].to(device)
        b_labels = batch[2].to(device)
        with torch.no_grad():
            pred = model(input_ids=b_input_ids, attention_mask=b_input_mask)
            yhat = pred.logits.argmax(axis=1)
            # Utilisation de .item() pour Ã©viter les fuites de mÃ©moire (memory leak) CUDA
            nbgood += (yhat == b_labels).sum().item()
            total += b_labels.size(0)
    acc = nbgood / total if total > 0 else 0
    return acc

%%

In [ ]:
tb = 25 # Batch Size
train_dataloader = DataLoader(dataset_train, batch_size=tb, shuffle=True)
test_dataloader = DataLoader(dataset_test, batch_size=tb, shuffle=False)

In [ ]:
nbepochs = 5
loss_fn = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)

Training Loop

In [ ]:
print("Starting Fine-tuning...")
for e in range(nbepochs):
    model.train()
    running_loss = 0.0
    
    for idx, batch in enumerate(train_dataloader):
        b_input_ids = batch[0].to(device)
        b_input_mask = batch[1].to(device)
        b_labels = batch[2].to(device)
        
        optimizer.zero_grad()
        
        # Forward pass
        pred = model(input_ids=b_input_ids, attention_mask=b_input_mask)
        loss = loss_fn(pred.logits, b_labels)
        
        # Backward pass
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        
        if idx > 0 and idx % 250 == 0:
            print(f"Epoch {e+1} | Batch {idx} | Loss: {running_loss / (idx+1):.4f}")
    
    # Validation Ã  la fin de chaque Epoch
    acc_train = accuracy(model, train_dataloader)
    acc_test = accuracy(model, test_dataloader)
    print(f"**** EPOCH {e+1} COMPLETED | Acc Train: {acc_train*100:.2f}% | Acc Test: {acc_test*100:.2f}% ****\n")